#Section-B

##Question B1

In [ ]:
corpus = """
Artificial Intelligence is a field of computer science that focuses on creating
systems capable of performing tasks that normally require human intelligence.
These tasks include learning, reasoning, problem solving and decision making.

Machine Learning is a branch of Artificial Intelligence. It allows computers
to learn patterns from data without being explicitly programmed for every task.
Common types of machine learning include supervised learning, unsupervised
learning and reinforcement learning.

Natural Language Processing enables computers to understand, process and
generate human language. Applications of NLP include chatbots, translation,
sentiment analysis, text classification and question answering.

Large Language Models are deep learning models trained on huge amounts of text.
They can generate text, answer questions, summarize documents and perform
many language-related tasks. Examples include GPT-based models and other
transformer-based language models.

Retrieval Augmented Generation combines information retrieval with language
generation. Instead of depending only on information stored in a language
model, RAG retrieves relevant documents and provides them as context to the
language model before generating an answer.

Embeddings are numerical representations of text. Similar texts tend to have
similar vector representations. Embeddings are commonly used for semantic
search, document retrieval, recommendation systems and RAG applications.
"""


In [14]:
# Fixed-size chunking
def chunking(text, size=250, overlap=50):
    chunks = []
    start = 0

    while start < len(text):
        end = start + size
        chunk = text[start:end]
        chunks.append(chunk.strip())

        if end >= len(text):
            break

        start = end - overlap

    return chunks


chunks = chunking(corpus, chunk_size=300, overlap=50)

print("Number of chunks:", len(chunks))

for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i+1} ---")
    print(chunk)

Number of chunks: 6

--- Chunk 1 ---
Artificial Intelligence is a field of computer science that focuses on creating
systems capable of performing tasks that normally require human intelligence.
These tasks include learning, reasoning, problem solving and decision making.

Machine Learning is a branch of Artificial Intelligence. It al

--- Chunk 2 ---
ning is a branch of Artificial Intelligence. It allows computers
to learn patterns from data without being explicitly programmed for every task.
Common types of machine learning include supervised learning, unsupervised
learning and reinforcement learning.

Natural Language Processing enables comput

--- Chunk 3 ---
rning.

Natural Language Processing enables computers to understand, process and
generate human language. Applications of NLP include chatbots, translation,
sentiment analysis, text classification and question answering.

Large Language Models are deep learning models trained on huge amounts of text

--- Chunk 4 ---
ep learnin

##Question B2

In [3]:
!pip install -q sentence-transformers scikit-learn

In [18]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Generate embeddings for all chunks
cembed = model.encode(chunks)

print("Embedding shape:", cembed.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding shape: (6, 384)


In [19]:
def dense_retrieval(query, chunks, cembed, top_k=3):

    # Convert query into embedding
    qembed = model.encode([query])

    # Calculate cosine similarity
    similarities = cosine_similarity(
        qembed,
        cembed
    )[0]

    # Get indices of top-k results
    top_indices = np.argsort(similarities)[::-1][:top_k]

    results = []

    for idx in top_indices:
        results.append({
            "chunk": chunks[idx],
            "score": similarities[idx]
        })

    return results

In [20]:
query1 = "How does RAG retrieve information?"

results1 = dense_retrieval(
    query1,
    chunks,
    cembed,
    top_k=3
)

print("Query:", query1)

for i, result in enumerate(results1):
    print(f"\nRank {i+1}")
    print("Similarity:", round(result["score"], 4))
    print(result["chunk"])

Query: How does RAG retrieve information?

Rank 1
Similarity: 0.4247
nted Generation combines information retrieval with language
generation. Instead of depending only on information stored in a language
model, RAG retrieves relevant documents and provides them as context to the
language model before generating an answer.

Embeddings are numerical representations of

Rank 2
Similarity: 0.2553
wer.

Embeddings are numerical representations of text. Similar texts tend to have
similar vector representations. Embeddings are commonly used for semantic
search, document retrieval, recommendation systems and RAG applications.

Rank 3
Similarity: 0.2026
ep learning models trained on huge amounts of text.
They can generate text, answer questions, summarize documents and perform
many language-related tasks. Examples include GPT-based models and other
transformer-based language models.

Retrieval Augmented Generation combines information retrieval wit


In [21]:
query2 = "What are numerical representations of text?"

results2 = dense_retrieval(
    query2,
    chunks,
    chunk_embeddings,
    top_k=3
)

print("Query:", query2)

for i, result in enumerate(results2):
    print(f"\nRank {i+1}")
    print("Similarity:", round(result["score"], 4))
    print(result["chunk"])

Query: What are numerical representations of text?

Rank 1
Similarity: 0.5875
wer.

Embeddings are numerical representations of text. Similar texts tend to have
similar vector representations. Embeddings are commonly used for semantic
search, document retrieval, recommendation systems and RAG applications.

Rank 2
Similarity: 0.3987
nted Generation combines information retrieval with language
generation. Instead of depending only on information stored in a language
model, RAG retrieves relevant documents and provides them as context to the
language model before generating an answer.

Embeddings are numerical representations of

Rank 3
Similarity: 0.383
rning.

Natural Language Processing enables computers to understand, process and
generate human language. Applications of NLP include chatbots, translation,
sentiment analysis, text classification and question answering.

Large Language Models are deep learning models trained on huge amounts of text


##Question B3


In [22]:
!pip install -q rank_bm25

In [9]:
from rank_bm25 import BM25Okapi

# Tokenize chunks
tokenized_chunks = [
    chunk.lower().split()
    for chunk in chunks
]

# Create BM25 index
bm25 = BM25Okapi(tokenized_chunks)

In [11]:
def bm25_retrieval(query, chunks, bm25, top_k=3):

    query_tokens = query.lower().split()

    scores = bm25.get_scores(query_tokens)

    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for idx in top_indices:
        results.append({
            "chunk": chunks[idx],
            "score": scores[idx]
        })

    return results

In [23]:
bm25_results1 = bm25_retrieval(
    query1,
    chunks,
    bm25,
    top_k=3
)

print("Query:", query1)

for i, result in enumerate(bm25_results1):
    print(f"\nRank {i+1}")
    print("BM25 Score:", round(result["score"], 4))
    print(result["chunk"])

Query: How does RAG retrieve information?

Rank 1
BM25 Score: 0.659
wer.

Embeddings are numerical representations of text. Similar texts tend to have
similar vector representations. Embeddings are commonly used for semantic
search, document retrieval, recommendation systems and RAG applications.

Rank 2
BM25 Score: 0.5688
nted Generation combines information retrieval with language
generation. Instead of depending only on information stored in a language
model, RAG retrieves relevant documents and provides them as context to the
language model before generating an answer.

Embeddings are numerical representations of

Rank 3
BM25 Score: 0.0
ep learning models trained on huge amounts of text.
They can generate text, answer questions, summarize documents and perform
many language-related tasks. Examples include GPT-based models and other
transformer-based language models.

Retrieval Augmented Generation combines information retrieval wit


In [13]:
bm25_results2 = bm25_retrieval(
    query2,
    chunks,
    bm25,
    top_k=3
)

print("Query:", query2)

for i, result in enumerate(bm25_results2):
    print(f"\nRank {i+1}")
    print("BM25 Score:", round(result["score"], 4))
    print(result["chunk"])

Query: What are numerical representations of text?

Rank 1
BM25 Score: 1.6047
wer.

Embeddings are numerical representations of text. Similar texts tend to have
similar vector representations. Embeddings are commonly used for semantic
search, document retrieval, recommendation systems and RAG applications.

Rank 2
BM25 Score: 1.4943
nted Generation combines information retrieval with language
generation. Instead of depending only on information stored in a language
model, RAG retrieves relevant documents and provides them as context to the
language model before generating an answer.

Embeddings are numerical representations of

Rank 3
BM25 Score: 0.4156
Artificial Intelligence is a field of computer science that focuses on creating
systems capable of performing tasks that normally require human intelligence.
These tasks include learning, reasoning, problem solving and decision making.

Machine Learning is a branch of Artificial Intelligence. It al
